In [1]:
import datetime as dt
import numpy as np
import pandas as pd 
import json
from shapely.geometry import Point, LineString, MultiLineString
import geopandas

In [2]:
colors_d = {
    'Ligne 1': '#FFCD00', 
    'Ligne 2': '#003CA6', 
    'Ligne 3': '#837902', 
    'Ligne 4': '#CF009E', 
    'Ligne 5': '#FF7E2E', 
    'Ligne 6': '#6ECA97',
    'Ligne 7': '#FA9ABA', 
    'Ligne 8': '#E19BDF',
    'Ligne 9': '#B6BD00',
    'Ligne 10': '#C9910D',
    'Ligne 11': '#704B1C',
    'Ligne A': '#2b5439',
    'Ligne 12': '#007852',
    'Ligne B': '#1891c4',
    'Ligne 13': '#6EC4E8',
    'Ligne 14': '#62259D',
    'Ligne 3 bis': '#6EC4E8',
    'Ligne 7 bis': '#6ECA97',
    'Couloirs': '#777779',
    'Ligne 2 Sud': '#ab1f07',
    'Ligne 14 (ancienne)': '#135e12',
    'Voie des Fêtes et voie navette': '#049fb0',
    # Grand Paris Express, official IDFM values.
    'Ligne 15': '#B90845',
    'Ligne 16': '#F3A4BA',
    'Ligne 17': '#D5C900',
    'Ligne 18': '#00A88F',
}

In [3]:
today = dt.datetime.combine(dt.date.today(), dt.time(0))

# Read data

In [ ]:
file_path = 'data/raw_data/'
stations_file = 'stations_history.csv'
segments_file = 'segments_history.csv'
stations_planned_file = 'stations_planned.csv'
segments_planned_file = 'segments_planned.csv'

In [ ]:
stations = pd.read_csv(file_path+stations_file)
segments = pd.read_csv(file_path+segments_file)

# Lines that have not opened live in their own files rather than behind a
# "start_date is in the future" test. A date test would quietly promote line 15
# to built the first time anyone rebuilt after its opening date, asserting as
# fact something nobody had checked; carrying the flag with the provenance means
# a projection stays a projection until someone moves the rows across.
stations_planned = pd.read_csv(file_path+stations_planned_file)
segments_planned = pd.read_csv(file_path+segments_planned_file)

stations['planned'] = False
segments['planned'] = False
stations_planned['planned'] = True
segments_planned['planned'] = True

stations = pd.concat([stations, stations_planned], ignore_index=True)
segments = pd.concat([segments, segments_planned], ignore_index=True)

In [ ]:
stations['start_date'] = pd.to_datetime(stations['start_date'])
stations['end_date'] = pd.to_datetime(stations['end_date'])

In [ ]:
segments['start_date'] = pd.to_datetime(segments['start_date'])
segments['end_date'] = pd.to_datetime(segments['end_date'])

# One sentinel closes everything that is still open, built and projected alike.
# It used to be the day the notebook ran, which worked while the data stopped at
# the present: now that the timeline reaches into 2028, that date would close the
# built network in the middle of the run - every station gaining a projected line
# would blink out on the sentinel date - and would close a projected line before
# it opened, dropping it from the output altogether. So the sentinel sits one day
# past the last opening in the data, or on the run date, whichever is later.
horizon = max(today,
              stations['start_date'].max() + dt.timedelta(days=1),
              segments['start_date'].max() + dt.timedelta(days=1))

stations['end_date'] = stations['end_date'].fillna(horizon)
segments['end_date'] = segments['end_date'].fillna(horizon)

# Lines to json

In [ ]:
lines = segments['line'].unique()
lines

In [ ]:
paths = []

# Grouped by (line, planned), not by line alone: the union below melts every
# segment of a snapshot into one geometry, so built and projected track have to
# be separated before it rather than after.
for line in lines:
    df_line = segments[(segments['line']==line)]
    for planned in sorted(df_line['planned'].unique()):
        df = df_line[(df_line['planned']==planned)]

        # A line's shape changes when its own segments change - and also when a
        # station on it moves. Four stations were rebuilt on a new site, and
        # without their dates among the key dates the line keeps whatever
        # position the station had when the segment opened: line 2 held Victor
        # Hugo at its pre-1931 site, 311 m from the dot, for the rest of the
        # timeline. Feeding the station dates in cuts a snapshot at the move;
        # the merge below throws away the cuts that changed nothing.
        served = set(df['from_station'])|set(df['to_station'])
        moves = stations[stations['lineage'].isin(served)]
        span_start = df['start_date'].min()
        span_end = df['end_date'].max()
        key_dates = sorted(set(df['start_date'])|set(df['end_date'].dropna())|
                           set(moves['start_date'].dropna())|set(moves['end_date'].dropna()))
        key_dates = [d for d in key_dates if span_start <= d <= span_end]

        for i in range(len(key_dates)-1): 
            start_date = key_dates[i]
            end_date = key_dates[i+1]
            _df = df[(df['start_date']<=start_date)&(df['end_date']>start_date)]
            path = None
            for j in _df.index:
                from_station = _df.loc[j, 'from_station']
                to_station = _df.loc[j, 'to_station']
                station_from = stations.loc[(stations['lineage']==from_station)& 
                                            (stations['start_date']<=start_date)&
                                            (stations['end_date']>start_date)]
                station_to = stations.loc[(stations['lineage']==to_station)& 
                                          (stations['start_date']<=start_date)&
                                          (stations['end_date']>start_date)]
                segment = LineString([Point([station_from['longitude'].values[0], 
                                             station_from['latitude'].values[0]]), 
                                      Point([station_to['longitude'].values[0], 
                                             station_to['latitude'].values[0]])
                                        ])
                if path == None: 
                    path = segment
                else: 
                    path = path.union(segment)

            if path is None:
                continue

            # Most of the extra cuts leave the shape untouched - a station being
            # renamed does not move the track - so a snapshot that is contiguous
            # with the one before it and identical in shape just extends it.
            if (paths and paths[-1]['line'] == line and paths[-1]['planned'] == bool(planned)
                    and paths[-1]['end_date'] == start_date
                    and paths[-1]['geometry'].equals(path)):
                paths[-1]['end_date'] = end_date
                continue

            paths.append({'line': line, 
                          'color': colors_d[line], 
                          'geometry': path, 
                          'start_date': start_date,
                          'end_date': end_date,
                          'planned': bool(planned)}
                         )

In [ ]:
lines_history = geopandas.GeoDataFrame(paths)
lines_history['start_date'] = lines_history['start_date'].map(lambda x: str(x.date()))
lines_history['end_date'] = lines_history['end_date'].map(lambda x: str(x.date()))

In [ ]:
lines_history.to_file("data/lines_history.geojson", driver='GeoJSON')

# Stations to json

In [ ]:
lineages = stations['lineage'].unique()

In [ ]:
station_history = []

for lineage in lineages: 
    lineage_records = stations[stations['lineage'] == lineage]
    
    lineage_segments = segments[
        (segments['from_station']==lineage)
        ].groupby(['start_date', 'end_date'])['line'].unique().reset_index()
    
    key_dates = sorted(set(lineage_records['start_date'])| 
                   set(lineage_records['end_date'])|
                   set(lineage_segments['start_date'])|
                   set(lineage_segments['end_date']))

    for i in range(len(key_dates)-1): 
            start_date = key_dates[i]
            end_date = key_dates[i+1]
            period_records = lineage_records.loc[
                (lineage_records['start_date']<=start_date)& 
                (lineage_records['end_date']>start_date)]
            if len(period_records):
                name = period_records['name'].values[0]
                latitude = period_records['latitude'].values[0]
                longitude = period_records['longitude'].values[0]
                planned = bool(period_records['planned'].values[0])
                geometry = Point([longitude, latitude])
                station_lines = lineage_segments.loc[(lineage_segments['start_date']<=start_date)& 
                                                     (lineage_segments['end_date']>start_date),
                                                     'line'].values
                station_lines = sorted(set([item for sublist in station_lines for item in sublist]))
                if len(station_lines)==1: 
                    color = colors_d[station_lines[0]] 
                else:
                    color = '#D8D8B9'
                    
                station_step = {
                    'start_date': start_date,
                    'end_date': end_date, 
                    'name': name, 
                    'geometry': geometry, 
                    'lines': station_lines,
                    'color': color, 
                    'planned': planned, 
                    'lineage': lineage
                }

                if len(station_history)>1:
                    if (station_step['name'] == station_history[-1]['name'] and
                        station_step['lines'] == station_history[-1]['lines'] and 
                        station_step['geometry'] == station_history[-1]['geometry'] and
                        station_step['start_date'] == station_history[-1]['end_date']):
                        station_history[-1]['end_date'] = end_date 
                    else: 
                        station_history.append(station_step)
                else:
                    station_history.append(station_step)

In [15]:
station_history = geopandas.GeoDataFrame(station_history)

In [ ]:
l = []
index_to_drop = []
for i in station_history.index:
    name_i = station_history.at[i, 'name']
    lineage_i = station_history.at[i, 'lineage']
    start_date_i = station_history.at[i, 'start_date']
    for j in station_history.index: 
        if j>i: 
            name_j = station_history.at[j, 'name']
            lineage_j = station_history.at[j, 'lineage']
            start_date_j = station_history.at[j, 'start_date']
            if (name_j == name_i and 
                lineage_j != lineage_i and
                start_date_j == start_date_i): 
                    l.append({'start_date': start_date_i, 'lineages': [lineage_j, lineage_i]})
                    index_to_drop.append(i)
                    index_to_drop.append(j)

In [ ]:
events = []
for event in l: 
    start_date = event['start_date']
    lineage_i = event['lineages'][0]
    lineage_j = event['lineages'][1]
    events_i = station_history[(station_history['lineage']==lineage_i)&
                               (station_history['start_date']>=start_date)]
    events_j = station_history[(station_history['lineage']==lineage_j)&
                               (station_history['start_date']>=start_date)]
    key_dates = sorted(set(events_i['start_date'])| 
                       set(events_i['end_date'])|
                       set(events_j['start_date'])|
                       set(events_j['end_date']))
    for i in range(len(key_dates)-1): 
        start_date = key_dates[i]
        end_date = key_dates[i+1]
        info_i = events_i.loc[
                (events_i['start_date']<=start_date)& 
                (events_i['end_date']>start_date)].to_dict(orient='records')[0]
        info_j = events_j.loc[
                (events_j['start_date']<=start_date)& 
                (events_j['end_date']>start_date)].to_dict(orient='records')[0]
        station_lines = info_i['lines'] + info_j['lines']
        station_lines = sorted(set(station_lines))
        
        for info in [info_i, info_j]: 
            info['lines'] = station_lines
            info['start_date'] = start_date
            info['end_date'] = end_date
            if len(station_lines)==1:
                info['color'] = colors_d[station_lines[0]]
            else: 
                '#D8D8B9'
            events.append(info)

In [18]:
station_history = pd.concat([station_history.drop(index_to_drop), 
                             pd.DataFrame(events)]).reset_index(drop=True)

In [ ]:
station_history['lines'] = station_history['lines'].map(lambda x: str(x))
station_history['start_date'] = station_history['start_date'].map(lambda x: str(x.date()))
station_history['end_date'] = station_history['end_date'].map(lambda x: str(x.date()))

In [ ]:
station_history[station_history['name'] == 'Liège']

In [ ]:
geopandas.GeoDataFrame(station_history).to_file("data/stations_history.geojson", driver='GeoJSON')